In [1]:
%matplotlib widget

import os
import mne
import tqdm
import h5py
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from pyarrow.parquet import ParquetFile
from mne.io import concatenate_raws, read_raw_edf

from bscarlos.settings import OUTPUT_FOLDER
from bscarlos.settings import RAW_KISPI_DATA_FOLDER
from bscarlos.settings import PROCESSED_KISPI_DATA_FOLDER
from bscarlos.detector_v3 import ClusteringDetector_v3
from bscarlos.data_processor_kispi import PatientDataProcessor

# EEG Burst Suppression Analysis
Analyze the EEG signals (each 2.5s window) in such way: [Classification of EEG Signals Based on Image Representation of Statistical Features J. Ashford et al. 2019](https://rdcu.be/dTZbg)

In [23]:
raw_data_files = sorted(list(RAW_KISPI_DATA_FOLDER.glob('*.edf')))
raw_data_files = for files in 
processed_data_files = sorted(list(PROCESSED_KISPI_DATA_FOLDER.glob('*_6ch_merged.parquet')))
data_attributes = pd.read_csv(PROCESSED_KISPI_DATA_FOLDER / 'data_attributes_kispi.csv').set_index("unique_patient_id")

##  View Raw Data

In [24]:
raw_data_files

[WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE008_120418U-A.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE008_annotated1.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE021_150901U-C.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE021_annotated1.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE021_annotated2.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE026_140520Q-A.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/raw_kispi/SE026_annotated1.edf'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI

## View Processed Data

In [8]:
processed_data_files

[WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE008_a1_6ch_merged.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE021_a1_6ch_merged.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE021_a2_6ch_merged.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE026_a1_6ch_merged.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE038_a1_6ch_merged.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE038_a2_6ch_merged.parquet')]

## Mean & SD 
The sample **mean** and sample **standard deviation** of each signal (8 features).

## Skewness & Kurtosis 
The sample **skewness** and sample **kurtosis** of each signal (8 features).

## Maxima & Minima
The **maximum** and **minimum value** of each signal (8 features).

## Variance & Covariance
The sample **variances** of each signal, plus the sample **covariances** of all signal pairs (10 features).

## Eigenvalues
- The **eigenvalues** of the **covariance matrix** (4features).
- The **upper triangular elements** of the matrix logarithm of the covariance matrix. (10 features)

## FFT
- The **magnitude of the frequency** components of each signal, obtained using a **Fast Fourier Transform (FFT)** (300 features).
- The ***frequency values of the ten most energetic components*** of the FFT, for each signal (40 features).